# Process TRex Output

Notebook for converting TRex `.npz` tracking outputs into scaled `.csv` files.

## What this notebook does

1. Finds `.npz` files in a selected input directory.
2. Applies arena scaling corrections to position, speed, acceleration, derivatives, and border distance.
3. Exports one `.csv` per input file.
4. Returns a small processing summary for quick checking.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

## Configuration

Update `INPUT_DIR` to the folder containing TRex `.npz` files.

Set `SEARCH_SUBFOLDERS = True` if your `.npz` files are inside child folders such as `control/` or treatment subdirectories.


In [ ]:
# Folder containing TRex .npz files
INPUT_DIR = Path("/Volumes/LaCie/free_swimming/7dpf_TRex/adenosine_2.5mM/control")

# By default, CSV files are written next to the .npz files.
# Change this to another folder if you want all output CSVs in one place.
OUTPUT_DIR: Path | None = None

# Set to True to search one level of subfolders.
SEARCH_SUBFOLDERS = True

## Arena scaling

TRex assumes a default arena size. These values correct the exported tracking data to match the actual arena/video crop dimensions.


In [ ]:
@dataclass(frozen=True)
class ArenaScale:
    """Scale factors used to convert TRex-assumed dimensions to actual dimensions."""

    actual_length_cm: float = 14.0
    assumed_length_cm: float = 30.0
    actual_width_cm: float = 6.8
    assumed_width_cm: float = 15.0

    @property
    def x(self) -> float:
        return self.actual_length_cm / self.assumed_length_cm

    @property
    def y(self) -> float:
        return self.actual_width_cm / self.assumed_width_cm

    @property
    def average(self) -> float:
        return float(np.mean([self.x, self.y]))


SCALE = ArenaScale()

print(f"x scale: {SCALE.x:.4f}")
print(f"y scale: {SCALE.y:.4f}")
print(f"average scale: {SCALE.average:.4f}")

## Processing functions

In [ ]:
def find_npz_files(input_dir: Path, search_subfolders: bool = False) -> list[Path]:
    """Return sorted TRex .npz files from `input_dir`.

    If `search_subfolders` is True, searches one level down inside child folders.
    """
    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

    if search_subfolders:
        files = [
            file_path
            for subfolder in input_dir.iterdir()
            if subfolder.is_dir()
            for file_path in subfolder.glob("*.npz")
        ]
    else:
        files = list(input_dir.glob("*.npz"))

    return sorted(files)


def require_keys(npz_file: np.lib.npyio.NpzFile, required_keys: Iterable[str], file_path: Path) -> None:
    """Raise a helpful error if an expected TRex field is missing."""
    missing_keys = sorted(set(required_keys) - set(npz_file.files))
    if missing_keys:
        missing = ", ".join(missing_keys)
        raise KeyError(f"{file_path.name} is missing required TRex field(s): {missing}")


def trex_npz_to_dataframe(file_path: Path, scale: ArenaScale) -> pd.DataFrame:
    """Convert one TRex .npz file to a scaled pandas DataFrame."""
    required_keys = [
        "time",
        "frame",
        "X#wcentroid",
        "Y#wcentroid",
        "SPEED#wcentroid",
        "ACCELERATION#wcentroid",
        "missing",
        "VX",
        "VY",
        "AX",
        "AY",
        "ANGLE",
        "MIDLINE_OFFSET",
        "BORDER_DISTANCE#pcentroid",
    ]

    with np.load(file_path) as fish:
        require_keys(fish, required_keys, file_path)

        return pd.DataFrame(
            {
                "time": fish["time"],
                "frame": fish["frame"],
                "x": fish["X#wcentroid"] * scale.x,
                "y": fish["Y#wcentroid"] * scale.y,
                "speed": fish["SPEED#wcentroid"] * scale.average,
                "acceleration": fish["ACCELERATION#wcentroid"] * scale.average,
                # True = fish present in frame; False = missing/lost identity
                "mask": ~fish["missing"].astype(bool),
                "derivative_x": fish["VX"] * scale.x,
                "derivative_y": fish["VY"] * scale.y,
                "acceleration_x": fish["AX"] * scale.x,
                "acceleration_y": fish["AY"] * scale.y,
                # In radians; 0 = fish horizontal, looking right
                "angle": fish["ANGLE"],
                # Use cautiously for small fish
                "tail_angle": fish["MIDLINE_OFFSET"],
                "border_distance": fish["BORDER_DISTANCE#pcentroid"] * scale.average,
            }
        )


def output_csv_path(npz_file: Path, output_dir: Path | None = None) -> Path:
    """Return the CSV path for one input .npz file."""
    if output_dir is None:
        return npz_file.with_suffix(".csv")

    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / f"{npz_file.stem}.csv"


def process_trex_files(
    input_dir: Path,
    scale: ArenaScale,
    output_dir: Path | None = None,
    search_subfolders: bool = False,
) -> pd.DataFrame:
    """Process all matching TRex .npz files and return a summary table."""
    npz_files = find_npz_files(input_dir, search_subfolders=search_subfolders)

    if not npz_files:
        raise FileNotFoundError(f"No .npz files found in: {input_dir}")

    summary_rows = []

    for npz_file in npz_files:
        fish_df = trex_npz_to_dataframe(npz_file, scale)
        csv_path = output_csv_path(npz_file, output_dir=output_dir)
        fish_df.to_csv(csv_path, index=False)

        summary_rows.append(
            {
                "input_file": str(npz_file),
                "output_file": str(csv_path),
                "rows": len(fish_df),
                "present_frames": int(fish_df["mask"].sum()),
                "missing_frames": int((~fish_df["mask"]).sum()),
            }
        )

    return pd.DataFrame(summary_rows)

## Run conversion

In [ ]:
summary = process_trex_files(
    input_dir=INPUT_DIR,
    scale=SCALE,
    output_dir=OUTPUT_DIR,
    search_subfolders=SEARCH_SUBFOLDERS,
)

summary

## Preview one processed file

Use this cell to inspect the first exported CSV.


In [ ]:
first_csv = Path(summary.loc[0, "output_file"])
preview_df = pd.read_csv(first_csv)

preview_df.head()